# Unsupervised Learning: Clustering Beispiel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klar74/WS2026_lecture/blob/main/Vorlesung_10/VL10_Unsupervised_Clustering_Simple.ipynb)

**Lernziel:** Verstehen von K-Means Clustering ohne Labels

**Szenario:** Kundensegmentierung anhand von Alter und Ausgaben in einem synthetischen Datensatz.

**Lehrbeispiel:** Die Gruppen sind für die Simulation konstruiert. Daraus folgen keine allgemeinen Aussagen über Alter und Kaufverhalten realer Menschen.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Synthetische Kundendaten erstellen
np.random.seed(42)

# 3 versteckte Kundengruppen simulieren
# Gruppe 1: Junge Kunden, niedrige Ausgaben
gruppe1_alter = np.random.normal(25, 5, 50)
gruppe1_ausgaben = np.random.normal(30, 10, 50)

# Gruppe 2: Mittleres Alter, mittlere Ausgaben  
gruppe2_alter = np.random.normal(40, 8, 50)
gruppe2_ausgaben = np.random.normal(70, 15, 50)

# Gruppe 3: Ältere Kunden, hohe Ausgaben
gruppe3_alter = np.random.normal(55, 7, 50)
gruppe3_ausgaben = np.random.normal(120, 20, 50)

# Alle Daten kombinieren
alter = np.concatenate([gruppe1_alter, gruppe2_alter, gruppe3_alter])
ausgaben = np.concatenate([gruppe1_ausgaben, gruppe2_ausgaben, gruppe3_ausgaben])

# DataFrame erstellen
df = pd.DataFrame({
    'alter': alter.round(0).astype(int),
    'ausgaben_euro': ausgaben.round(2)
})

print(f"📊 Kundendaten: {len(df)} Kunden")
print(f"Alter: {df['alter'].min()} - {df['alter'].max()} Jahre")
print(f"Ausgaben: {df['ausgaben_euro'].min():.2f} - {df['ausgaben_euro'].max():.2f} €")

df.head()

In [ ]:
# Daten visualisieren - noch ohne Cluster
plt.figure(figsize=(10, 6))
plt.scatter(df['alter'], df['ausgaben_euro'], alpha=0.7, s=50, color='gray')
plt.xlabel('Alter (Jahre)')
plt.ylabel('Ausgaben (€)')
plt.title('Kundendaten - Versteckte Gruppen?')
plt.grid(True, alpha=0.3)
plt.show()

print("🤔 Frage: Siehst du versteckte Gruppen in den Daten?")
print("💡 K-Means kann ähnliche Punkte gruppieren. Die Anzahl k müssen wir vorgeben!")

## K-Means: Was wird gelernt, was geben wir vor?

Wir setzen **k=3 als Lehrannahme**. K-Means lernt Zentren und Zuordnungen, nicht automatisch die richtige Anzahl von Gruppen. Die für die Simulation bekannten Gruppenzugehörigkeiten werden dem Algorithmus nicht übergeben.

K-Means minimiert quadrierte euklidische Abstände im verwendeten Merkmalsraum und passt besonders zu kompakten, ähnlich großen Gruppen. Standardisierung verhindert hier, dass die Euro-Skala allein wegen ihrer größeren Streuung dominiert. Sie ist eine bewusste Gewichtungsentscheidung, keine Garantie für sinnvolle Segmente.

**Cluster-IDs sind beliebig:** Cluster 0 bedeutet nicht automatisch jung oder sparsam. Beschreibungen müssen aus den gelernten Zentren stammen.

Wir beschreiben den vorhandenen Kundenbestand, daher ist hier kein Train/Test-Split erforderlich. Für spätere Kunden würde man den bereits gefitteten Scaler mit `transform` und K-Means mit `predict` anwenden, ohne beide jeweils neu zu fitten.

In [ ]:
# K-Means Clustering anwenden
print("🔍 K-MEANS CLUSTERING")
print("=" * 30)

# Daten standardisieren (wichtig für K-Means!)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[['alter', 'ausgaben_euro']])

# K-Means mit 3 Clustern
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

# Cluster-Labels zu DataFrame hinzufügen
df['cluster'] = cluster_labels

# Cluster-Zentren zurück transformieren
centers_original = scaler.inverse_transform(kmeans.cluster_centers_)

print(f"✅ Zuordnung zu {len(set(cluster_labels))} Clustern bei vorgegebenem k=3 berechnet")
print("\n📋 Cluster-Größen:")
cluster_counts = df['cluster'].value_counts().sort_index()
for cluster, count in cluster_counts.items():
    print(f"   Cluster {cluster}: {count} Kunden")

# Cluster-Zentren anzeigen
print("\n🎯 Cluster-Zentren:")
for i, center in enumerate(centers_original):
    print(f"   Cluster {i}: {center[0]:.1f} Jahre, {center[1]:.2f} €")

In [ ]:
# Ergebnisse visualisieren
plt.figure(figsize=(12, 5))

# Subplot 1: Original Daten
plt.subplot(1, 2, 1)
plt.scatter(df['alter'], df['ausgaben_euro'], alpha=0.7, s=50, color='gray')
plt.xlabel('Alter (Jahre)')
plt.ylabel('Ausgaben (€)')
plt.title('Vorher: Unbekannte Struktur')
plt.grid(True, alpha=0.3)

# Subplot 2: Mit Clustern
plt.subplot(1, 2, 2)
colors = ['red', 'blue', 'green']
cluster_names = {i: f'{center[0]:.0f} Jahre / {center[1]:.0f} €' for i, center in enumerate(centers_original)}

for i in range(3):
    cluster_data = df[df['cluster'] == i]
    plt.scatter(cluster_data['alter'], cluster_data['ausgaben_euro'], 
               alpha=0.7, s=50, color=colors[i], label=f'Cluster {i}: {cluster_names[i]}')

# Cluster-Zentren markieren
plt.scatter(centers_original[:, 0], centers_original[:, 1], 
           marker='x', s=200, linewidths=3, color='black', label='Zentren')

plt.xlabel('Alter (Jahre)')
plt.ylabel('Ausgaben (€)')
plt.title('Nachher: K-Means-Zuordnung bei k=3')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🎉 UNSUPERVISED LEARNING ERFOLGREICH!")
print("✅ Ohne Ziel-Labels Zentren und Zuordnungen für vorgegebenes k=3 gelernt")
print("Die Beschreibungen zeigen Cluster-Mittelwerte, keine Eigenschaften jeder einzelnen Person.")
print("Geschäftlicher Nutzen und Stabilität der Segmente müssen gesondert geprüft werden.")

## Wie viele Cluster sind sinnvoll?

Wir vergleichen einige Werte von k auf demselben standardisierten Datensatz:
- **Inertia:** Summe der quadrierten Abstände zum zugeordneten Zentrum. Sie sinkt typischerweise mit k; nur den kleinsten Wert zu wählen würde immer mehr Cluster belohnen. Die Elbow-Methode sucht einen Knick, liefert aber nicht immer eine eindeutige Antwort.
- **Silhouette:** Vergleicht die Nähe zum eigenen Cluster mit der zum nächsten anderen Cluster. Werte nahe 1 sprechen für kompakte, getrennte Gruppen; negative Werte für ungünstige Zuordnungen. Der Wert ist keine Klassifikationsgenauigkeit und kein Beweis für wahre Kundentypen.

Auch wenn die Simulation drei Gruppen erzeugt hat, kann ein internes Kriterium ein anderes k bevorzugen. Wir behalten k=3 zur Illustration bei und ändern die Zuordnung oben nicht nachträglich. Bei echten Daten gehören fachlicher Nutzen und Stabilität über Stichproben und Initialisierungen zur Entscheidung.

In [ ]:
diagnostic_rows = []
for k in range(2, 7):
    candidate = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = candidate.fit_predict(X_scaled)
    diagnostic_rows.append({'k': k, 'Inertia': candidate.inertia_, 'Silhouette': silhouette_score(X_scaled, labels)})
cluster_diagnostics = pd.DataFrame(diagnostic_rows).set_index('k')
print(cluster_diagnostics.round(3))
print(f"Höchster Silhouette-Wert unter den geprüften k: {cluster_diagnostics['Silhouette'].idxmax()}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(cluster_diagnostics.index, cluster_diagnostics['Inertia'], marker='o')
axes[0].set(xlabel='Anzahl Cluster k', ylabel='Inertia', title='Elbow-Diagnose')
axes[1].plot(cluster_diagnostics.index, cluster_diagnostics['Silhouette'], marker='o')
axes[1].set(xlabel='Anzahl Cluster k', ylabel='Silhouette', title='Interne Trennung der Cluster')
for ax in axes:
    ax.set_xticks(cluster_diagnostics.index)
plt.tight_layout()
plt.show()

## 💡 Was haben wir gelernt?

**Unsupervised Learning:**
- 🔍 **Keine Ziel-Labels nötig** - K-Means lernt Zuordnungen und Zentren, aber k wird vorgegeben
- 🎯 **K-Means Clustering** - Gruppiert ähnliche Datenpunkte im gewählten Merkmalsraum
- 📊 **Standardisierung wichtig** - Skalen und damit die Gewichtung der Merkmale bewusst festlegen
- Cluster-IDs haben keine feste Bedeutung; Beschreibungen aus Zentren und Verteilungen ableiten
- Interne Gütemaße unterstützen die Wahl von k, beweisen aber keine natürlichen Kundentypen

**Business Value:**
- 👥 **Kundensegmentierung** als explorative Beschreibung, abhängig von Features, Skalierung und k
- 🎯 Mögliche Marketing-Hypothesen anschließend fachlich und empirisch prüfen
- 💰 Geschäftlicher Mehrwert folgt nicht automatisch aus einer Clusterzuordnung

**Nächste Schritte:**
- Clusterzahl mit mehreren Kriterien und fachlicher Plausibilität beurteilen
- Andere Clustering-Algorithmen (DBSCAN, Hierarchical)
- Cluster-Charakteristika analysieren